In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Mandir_Marg_Delhi_DPCC_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,362.0,NaN,159.0,118.0,141.0,152.0,63.0,NaN,78.0,135.0,348.0,258.0
1,2,343.0,NaN,135.0,119.0,152.0,131.0,69.0,47.0,65.0,NaN,321.0,274.0
2,3,327.0,NaN,122.0,128.0,NaN,110.0,68.0,52.0,NaN,NaN,384.0,249.0
3,4,376.0,NaN,122.0,126.0,NaN,151.0,63.0,53.0,53.0,115.0,389.0,137.0
4,5,352.0,NaN,127.0,112.0,NaN,170.0,56.0,53.0,55.0,88.0,370.0,127.0
5,6,327.0,152.0,132.0,113.0,NaN,144.0,56.0,56.0,NaN,80.0,350.0,164.0
6,7,348.0,135.0,158.0,98.0,NaN,182.0,59.0,56.0,59.0,81.0,389.0,202.0
7,8,365.0,166.0,139.0,112.0,NaN,175.0,NaN,55.0,NaN,129.0,375.0,245.0
8,9,360.0,123.0,137.0,116.0,NaN,121.0,66.0,56.0,94.0,133.0,358.0,187.0
9,10,291.0,296.0,168.0,139.0,NaN,114.0,94.0,55.0,83.0,94.0,333.0,239.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,362.000000,177.344828,159.0,118.000000,141.000000,152.0,63.000000,52.03125,78.000000,135.000000,348.000000,258.000000
1,2,343.000000,177.344828,135.0,119.000000,152.000000,131.0,69.000000,47.00000,65.000000,178.212121,321.000000,274.000000
2,3,327.000000,177.344828,122.0,128.000000,124.210526,110.0,68.000000,52.00000,69.517241,178.212121,384.000000,249.000000
3,4,376.000000,177.344828,122.0,126.000000,124.210526,151.0,63.000000,53.00000,53.000000,115.000000,389.000000,137.000000
4,5,352.000000,177.344828,127.0,112.000000,124.210526,113.9,56.000000,53.00000,55.000000,88.000000,370.000000,127.000000
5,6,327.000000,152.000000,132.0,113.000000,124.210526,144.0,56.000000,56.00000,69.517241,80.000000,350.000000,164.000000
6,7,348.000000,135.000000,158.0,98.000000,124.210526,113.9,59.000000,56.00000,59.000000,81.000000,389.000000,202.000000
7,8,365.000000,166.000000,139.0,112.000000,124.210526,113.9,70.088235,55.00000,69.517241,129.000000,375.000000,245.000000
8,9,360.000000,123.000000,137.0,116.000000,124.210526,121.0,66.000000,56.00000,94.000000,133.000000,358.000000,187.000000
9,10,291.000000,296.000000,168.0,139.000000,124.210526,114.0,94.000000,55.00000,83.000000,94.000000,333.000000,239.000000
